# Raw Data Preprocessor
This notebook cleans and prepares the raw data straight from the sensor to the desired format.

In [9]:
import os
import re
import pandas as pd
import shutil

In [10]:
#Declarations
DATE = '2026-06-24'
META_FILE_PATH = f'../data/00_metadata/{DATE}.csv'
RAW_FOLDER_PATH = f'../data/01_raw/{DATE}'
PROC_FOLDER_PATH = f"../data/02_preprocessed_test/{DATE}"
SHIFT_INDEX = -1                # = [1 - {FILE SEQ}]
INIT_TRIAL_NO_SHIFT = 0

In [11]:
if os.path.exists(PROC_FOLDER_PATH):                           # Delete the existing folder and all its contents
    shutil.rmtree(PROC_FOLDER_PATH)
os.makedirs(PROC_FOLDER_PATH)                                  # Create a new empty folder
files = os.listdir(RAW_FOLDER_PATH)                            # Create a list containing the file names of the raw files

csv_files = [
    file
    for file in files
    if ".csv" in file.lower()
    #if file.lower().endswith(".csv_files")
]

csv_files = sorted(
    csv_files,
    key=lambda file: int(
        re.search(r"\((\d+)\)", file).group(1)
    )
)

In [12]:
for c in csv_files:
    match = re.search(r"\((\d+)\)", c)
    #print(match)

In [13]:
# Get run number to match with experiment logs
run_index = {}

for c in csv_files:
    match = re.search(r"\((\d+)\)", c)
    run_no = int(match.group(1)) if match else None
    run_no = run_no + SHIFT_INDEX
    run_index[run_no] = c

run_index = dict(sorted(run_index.items()))

In [14]:
metadata = pd.read_csv(META_FILE_PATH)
metadata['File'] = metadata['Run_no'].apply(lambda x: run_index[x])
#metadata.head(51)

In [15]:
metadata

,DATE,RH_percent,Run_no,Material,k,Mass,Volume,rho,cp,Trial,Material_T0_C,Sensor_T0_C,Timestamp,Valid,File
0,2026/06/24,43-44,1,carbon,8.500,1,1,2300,717,1,40.3,24.4,15:48:27,False,I_V-t Sampling [(2) ; 2026_06_24 15_48_27].csv
1,2026/06/24,43-44,2,carbon,8.500,1,1,2300,717,2,37.9,23.9,15:52:50,False,I_V-t Sampling [(3) ; 2026_06_24 15_52_50].csv
2,2026/06/24,43-44,3,carbon,8.500,1,1,2300,717,1,40.4,24.7,16:13:36,True,I_V-t Sampling [(4) ; 2026_06_24 16_13_36].csv
3,2026/06/24,43-44,4,carbon,8.500,1,1,2300,717,2,40.2,24.6,16:16:19,True,I_V-t Sampling [(5) ; 2026_06_24 16_16_19].csv
4,2026/06/24,43-44,5,carbon,8.500,1,1,2300,717,3,39.9,24.5,16:18:27,True,I_V-t Sampling [(6) ; 2026_06_24 16_18_27].csv
5,2026/06/24,43-44,6,gypsum,0.170,1,1,2300,1090,1,40.0,24.6,16:21:01,True,I_V-t Sampling [(7) ; 2026_06_24 16_21_01].csv
6,2026/06/24,43-44,7,gypsum,0.170,1,1,2300,1090,2,39.9,24.8,16:23:10,True,I_V-t Sampling [(8) ; 2026_06_24 16_23_10].csv
7,2026/06/24,43-44,8,gypsum,0.170,1,1,2300,1090,3,39.9,24.4,16:25:33,True,I_V-t Sampling [(9) ; 2026_06_24 16_25_33].csv
8,2026/06/24,43-44,9,cement,1.730,1,1,2700,880,1,39.8,24.5,16:27:25,True,I_V-t Sampling [(10) ; 2026_06_24 16_27_25].csv
9,2026/06/24,43-44,10,cement,1.730,1,1,2700,880,2,39.9,24.8,16:28:49,True,I_V-t Sampling [(11) ; 2026_06_24 16_28_49].csv


In [16]:
max_trial = int(metadata["Trial"].max())
zero_pad = len(str(max_trial))


for index, row in metadata.iterrows():

    valid = row["Valid"]

    if valid:
        file = row["File"]

        material = (
            str(row["Material"])
            .strip()
            .lower()
            .replace(" ", "_")
        )

        trial_no = int(row["Trial"])
        therm_cond = float(row["k"])
        mass = float(row["Mass"])
        vol = float(row["Volume"])
        density = float(row["rho"])
        heatcap = float(row["cp"])

        file_no = trial_no + INIT_TRIAL_NO_SHIFT

        file_name = (
            f"{PROC_FOLDER_PATH}/{material}_"
            #f"{trial_no:0{zero_pad}d}.csv"
            f"{file_no:0{zero_pad}d}.csv"
        )

        df = pd.read_csv(
            f"{RAW_FOLDER_PATH}/{file}",
            skiprows=255
        )

        # Clean column names
        df.columns = df.columns.str.strip()

        # Keep and rename the required sensor columns
        df = (
            df[["Time", "I1", "I2"]]
            .rename(columns={
                "I1": "Primary",
                "I2": "Secondary"
            })
            .copy()
        )

        # Add experiment information
        df["Sample"] = material
        df["Trial"] = trial_no + INIT_TRIAL_NO_SHIFT
        df["k"] = therm_cond
        df["Mass"] = mass
        df["Volume"] = vol
        df["rho"] = density
        df["cp"] = heatcap

        # Clean and organize rows
        df = (
            df
            .dropna(subset=["Time", "Primary", "Secondary"])
            .sort_values("Time")
            .reset_index(drop=True)
        )

        # Create a new sequential index column
        df.insert(0, "index", range(len(df)))

        # Final column order
        df = df[
            [
                "index",
                "Sample",
                "Trial",
                "k",
                "Mass",
                "Volume",
                "rho",
                "cp",
                "Time",
                "Primary",
                "Secondary"
            ]
        ]

        df.to_csv(file_name, index=False)

    else:
        print(row)

DATE                                                 2026/06/24
RH_percent                                                43-44
Run_no                                                        1
Material                                                 carbon
k                                                           8.5
Mass                                                          1
Volume                                                        1
rho                                                        2300
cp                                                          717
Trial                                                         1
Material_T0_C                                              40.3
Sensor_T0_C                                                24.4
Timestamp                                              15:48:27
Valid                                                     False
File             I_V-t Sampling [(2) ; 2026_06_24 15_48_27].csv
Name: 0, dtype: object
DATE             